#use analy environment

In [1]:
###what this script does:
    #start with config file for llama-1B-20BT-weightdecay{wd}-seed42-{sft_dataset}.yaml
    #change model path
    #change output dir
    #change run name
    #(don't need to change sft dataset since it's the same as the starting config)

In [2]:
import shutil
from itertools import product

In [3]:
##args -- write new config files with this setup
sft_dataset = 'medmcqa' #options: ['hellaswag', 'medmcqa', 'pubmedqa', 'mmluprocot', 'metamathqa', 'race'] #did not do hellaswag

### write config files for SFT, based on olmo model configs

In [4]:
model_size_lst = ['llama-4B-80BT']
wd_pretrain_lst = [0.1] #[0.1, 1.0]

model_path_dict = {
    'llama-4B-80BT-weightdecay0.1': '/n/home07/than157/desktop/done-large_projects/learn-better/evolm/models/hf_ckpts/zhenting/myllama-4B-80BT',
    #### TO DO: add path for llama-4B-80BT-weightdecay1.0
    'llama-4B-80BT-weightdecay1.0': '/n/home07/than157/desktop/done-large_projects/learn-better/evolm/models/hf_ckpts/XXX/XXXXX',
}


In [ ]:
n_newconfig_files_created = 0

for model_size, wd_pretrain in product(model_size_lst, wd_pretrain_lst):

    #make a copy of config file for metamathqa
    print(f"Creating config file for: model {model_size}, wd_pretrain {wd_pretrain}, sft_dataset {sft_dataset}")
    og_config_file_path = f'config_hub/custom_configs/ft_{sft_dataset}/llama-1B-20BT-weightdecay{wd_pretrain}-seed42-{sft_dataset}.yaml'
    new_config_file_path = f'config_hub/custom_configs/ft_{sft_dataset}/{model_size}-weightdecay{wd_pretrain}-seed42-{sft_dataset}.yaml'
    shutil.copyfile(og_config_file_path, new_config_file_path)

    ### make edits to the new config file
    with open(new_config_file_path, "r", encoding="utf-8") as f:
        text = f.read()

        #change every occurrence of "metamathqa" to "hellaswag" (for example)
            #changes these variables in config file:
            # model_name_or_path: llama-4B has different directory than llama-1B (since the former is downloaded and the later I pretrained)
            # output_dir
            # run_name
            ### change the following based on example config file in config_hub/custom_configs/4B/example.yaml
            #per_device_train_batch_size
            #gradient_accumulation_steps
            #learning_rate

        #change model_name_or_path
        old_str = "model_name_or_path: /n/home07/than157/desktop/done-large_projects/learn-better/evolm/pretrain/lit-trainer/models/pretrained/llama-1B-20BT-weightdecay0.1-seed42/final-hf"
        key = f"{model_size}-weightdecay{wd_pretrain}"
        new_str = f"model_name_or_path: {model_path_dict[key]}"
        text = text.replace(old_str, new_str)

        #change other appearances of model_size -- effectively changing output_dir and run_name
        text = text.replace("llama-1B-20BT", model_size)

        #change other variables based on example config file in config_hub/custom_configs/4B/example.yaml
        old_str = 'per_device_train_batch_size: 16'
        new_str = 'per_device_train_batch_size: 4'
        text = text.replace(old_str, new_str)

        old_str = 'gradient_accumulation_steps: 1'
        new_str = 'gradient_accumulation_steps: 4'
        text = text.replace(old_str, new_str)

        old_str = 'learning_rate: 1.0e-5'
        new_str = 'learning_rate: 7.5e-6'
        text = text.replace(old_str, new_str)
        
        #write new config file
        with open(new_config_file_path, "w", encoding="utf-8") as f:
            f.write(text)

    n_newconfig_files_created += 1

print(n_newconfig_files_created)
print("Complete!")


Creating config file for: model llama-4B-80BT, wd_pretrain 0.1, sft_dataset medmcqa
1
Complete!
